> **Disclaimer.** This project is educational and is not investment advice. Backtest and optimizer results are historical and do not predict future results.

# 123 de compra: backtest and optimization

One ticker, one strategy, from candles to charts. The rules live in `docs/brief.md`, section 7; this notebook only sets parameters and calls `b3core` (engine, metrics, optimizer) and `b3report` (charts). It holds no trading logic.

**Before trusting any number here:**
- Candle times are assumed to be `America/Sao_Paulo` wall-clock time, and it is not known whether the MT5 history is adjusted for dividends and splits (brief, section 10).
- Several engine details are assumptions waiting for sign-off (brief, section 7).
- `score_v1` is not implemented yet: its formula needs approval first, so the objectives below are result, profit factor, EV and drawdown.
- 123 de venda is not specified yet. Day trade is specified (section 6 below), but its forced exit is a stopgap unless finer candles are supplied.

In [ ]:
from collections import Counter
from dataclasses import replace
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd

from b3core.data import CsvProvider, Timeframe
from b3core.engine import BacktestConfig, run_backtest
from b3core.optimizer import (
    Constraints,
    GridSearch,
    ParamRange,
    ParamSpace,
    RandomSearch,
    describe_space,
    optimize,
)
from b3report.charts import (
    equity_curve_figure,
    heatmap_figure,
    monthly_returns,
    monthly_returns_figure,
    trade_chart,
)

pd.options.display.float_format = "{:,.4f}".format

## 1. Parameters

Everything the run depends on is a plain variable here. The CSV folder is gitignored vendor data: never commit it.

In [ ]:
DATA_DIR = Path("../samples")  # MT5 "export bars" files, gitignored
SYMBOL = "PETR4"
TIMEFRAME = Timeframe.H2  # D1, H2 or H4 with the current samples
START = datetime(2016, 1, 1, tzinfo=UTC)
END = datetime(2027, 1, 1, tzinfo=UTC)  # exclusive

CONFIG = BacktestConfig(
    capital_brl=100_000,
    max_risk_brl=2_000,
    tick_size=0.01,
    lot_size=100,
    stop_mode="c2_low",  # "c2_low" or "c3_low"
    target_mode="risk_fixed",  # "risk_fixed" or "amplitude"
    risk_factor=2.0,
    entry_offset_ticks=1,
    entry_validity_candles=1,
    inside_bar=False,
)

## 2. Load candles

Problems found on ingest are listed, never silently fixed (brief, section 5).

In [ ]:
candles = CsvProvider(DATA_DIR).get_candles(SYMBOL, TIMEFRAME, START, END)
print(f"{SYMBOL} {TIMEFRAME.value}: {len(candles)} candles, {candles.ts.iloc[0]} to {candles.ts.iloc[-1]}")
print("ingest issues:", candles.attrs["issues"] or "none")
candles.head()

## 3. Check the engine against the worked example

The five candles of brief section 7. The expected result, worked out by hand, is a long of 500 shares at 10.56 exiting at the 11.28 target: **+R$ 360.00**, exactly 2.0 R.

In [ ]:
example = pd.DataFrame(
    {
        "ts": pd.date_range("2024-01-02 13:00", periods=5, freq="2h", tz="UTC"),
        "open": [10.20, 10.30, 10.50, 10.40, 10.55],
        "high": [10.40, 10.60, 10.55, 10.58, 11.30],
        "low": [10.10, 10.20, 10.35, 10.38, 10.50],
        "close": [10.30, 10.50, 10.40, 10.55, 11.25],
        "volume": 1000.0,
    }
)
check = run_backtest(example, BacktestConfig(max_risk_brl=200))
(trade,) = check.trades
print(f"entry {trade.entry:.2f}  exit {trade.exit:.2f}  qty {trade.qty}  pnl R$ {trade.pnl:.2f}  ({trade.r_multiple:.1f} R)")
assert round(trade.pnl, 2) == 360.00 and trade.qty == 500, "worked example failed"
trade_chart(example, trade, context=3)

## 4. Backtest

In [ ]:
result = run_backtest(candles, CONFIG)
metrics = result.metrics
pd.Series(metrics.to_dict(), name=f"{SYMBOL} {TIMEFRAME.value}").to_frame()

In [ ]:
trades = pd.DataFrame([vars(t) for t in result.trades])
trades.head(10)

In [ ]:
equity_curve_figure(result)

In [ ]:
monthly_returns_figure(result)

The best and the worst trade on the candle chart (stop and target as dashed lines).

In [ ]:
if result.trades:
    best = max(result.trades, key=lambda t: t.pnl)
    worst = min(result.trades, key=lambda t: t.pnl)
    display(trade_chart(candles, best), trade_chart(candles, worst))
else:
    print("no trades with these parameters")

In [ ]:
monthly_returns(result).tail(12)

## 5. Optimization

Each parameter is a fixed value, a `ParamRange(start, stop, step)` or a list of options. The optimizer reports the size of the space before running, searches **in-sample only**, then re-runs the best results **out-of-sample** and flags large drops (brief, section 12).

Read the answer with care: the best of many trials is partly luck, so always look at `n_trials` next to it, at the out-of-sample columns and at the heatmap.

In [ ]:
SPACE = ParamSpace(
    {
        "risk_factor": ParamRange(1, 4, 0.5),
        "entry_offset_ticks": ParamRange(0, 3, 1),
        "entry_validity_candles": ParamRange(1, 3, 1),
        "stop_mode": ["c2_low", "c3_low"],
        "target_mode": ["risk_fixed", "amplitude"],
    }
)
info = describe_space(SPACE)
print(f"search space: {info.size} combinations, recommended algorithm: {info.recommendation}")

In [ ]:
OBJECTIVE = "result"  # "result", "profit_factor", "ev" or "min_drawdown"
CONSTRAINTS = Constraints(min_trades=30)  # also: max_drawdown=0.25
ALGORITHM = GridSearch()  # or RandomSearch(seed=42, n_samples=100)
WORKERS = 1  # >1 evaluates trials in a process pool, same results

opt = optimize(
    candles,
    CONFIG,
    SPACE,
    objective=OBJECTIVE,
    constraints=CONSTRAINTS,
    algorithm=ALGORITHM,
    oos_fraction=0.3,  # last 30% of the candles are never used by the search
    top_n=5,
    workers=WORKERS,
)
best = opt.best
print(f"{opt.n_trials} trials. best in-sample {OBJECTIVE}: {best.objective:+.4f}")
print("best parameters:", best.params)

In-sample versus out-of-sample for the top results. `dropped` is True when the out-of-sample objective lost more than half of the in-sample one.

In [ ]:
opt.validation_table()

Top of the in-sample ranking (infeasible trials, those breaking the constraints, are ranked last).

In [ ]:
trials = opt.trials_table()
ranked = trials.sort_values(["feasible", "objective"], ascending=False)
ranked.drop(columns=["run_hash", "phase"]).head(10)

**Neighbourhood view.** A good point surrounded by bad neighbours is a warning sign; a broad plateau is a good sign. Cells average the parameters not shown.

In [ ]:
heatmap_figure(trials, x="entry_offset_ticks", y="risk_factor")

In [ ]:
heatmap_figure(trials, x="stop_mode", y="target_mode")

## 6. Day trade

Nothing is held overnight. Parameters (brief, section 7): `min_entry_time` and `max_entry_time` bound the candle that fills an entry (defaults: start of day, and the exit time), and `exit_time` (here 16:30) is the forced exit. The 123 pattern may use yesterday's candles, but the order only fills on the day of its signal.

**How the 16:30 exit is priced.** With finer candles (an M30 or M15 export loaded below) the exit is pinpointed: the open of the candle that starts at 16:30. Without them, the stopgap is the open of the candle that contains 16:30 (on H2, the 16:00 candle), which is slightly early and never uses later prices. Day trade needs H2 or finer: on H4 only a handful of candles can qualify as entries.

In [ ]:
DAY_CONFIG = replace(
    CONFIG,
    day_trade=True,
    min_entry_time="00:00",  # start of day
    max_entry_time=None,  # None means exit_time
    exit_time="16:30",  # end-of-day call
    day_exit_policy="auto",  # "auto": use finer candles if given; "last_candle_open": never
)

try:
    INTRADAY = CsvProvider(DATA_DIR).get_candles(SYMBOL, Timeframe.M30, START, END)
    print(f"finer candles: {len(INTRADAY)} M30 candles pinpoint the 16:30 exit")
except FileNotFoundError:
    INTRADAY = None
    print("no M30 file for this symbol: the exit uses the open of the candle that contains 16:30")

In [ ]:
day_result = run_backtest(candles, DAY_CONFIG, timeframe=TIMEFRAME, intraday=INTRADAY)
print("exit reasons:", dict(Counter(t.exit_reason for t in day_result.trades)))
pd.Series(day_result.metrics.to_dict(), name="day trade").to_frame()

In [ ]:
display(equity_curve_figure(day_result))
forced = [t for t in day_result.trades if t.exit_reason == "time"]
if forced:
    display(trade_chart(candles, forced[0]))

Optimization of the entry window. Combinations whose window is invalid (minimum not before the maximum, or a maximum after the exit time) are dropped before any backtest runs.

In [ ]:
DAY_SPACE = ParamSpace(
    {
        "risk_factor": ParamRange(1, 4, 0.5),
        "min_entry_time": ["00:00", "11:00"],
        "max_entry_time": [None, "13:00"],
    }
)
info = describe_space(DAY_SPACE, base=DAY_CONFIG)
print(f"search space: {info.size} combinations, recommended algorithm: {info.recommendation}")

day_opt = optimize(
    candles,
    DAY_CONFIG,
    DAY_SPACE,
    objective=OBJECTIVE,
    constraints=CONSTRAINTS,
    oos_fraction=0.3,
    top_n=5,
    timeframe=TIMEFRAME,
    intraday=INTRADAY,
)
print(f"{day_opt.n_trials} trials. best in-sample {OBJECTIVE}: {day_opt.best.objective:+.4f}")
print("best parameters:", day_opt.best.params)
day_opt.validation_table()

## 7. Next steps

- Genetic search behind the same `Optimizer` interface (brief, section 12), compared with the grid on a small space.
- 123 de venda, once its rules are written.
- An M30 or M15 export from MT5, to pinpoint the 16:30 exit instead of the stopgap.
- `score_v1` after its formula is approved in `docs/score.md`.